# Lab: glacier velocity from image pairs

```{note}
This lab accompanies {doc}`panchromatic-imagery`. The parent chapter explains why cross-correlation works and what error sources limit it; here you implement the method from scratch, run it on one real image pair, and then confront the gap between one pair and the continental velocity mosaics the modeling chapters rely on.
```

The core idea is almost embarrassingly simple. A crevasse sits at some pixel in an image taken on day one. Six months later the same crevasse sits a few pixels over in a new image, because the ice carried it. Find the shift and you have the displacement. Divide by time and you have velocity. Everything that follows is disciplined bookkeeping around that shift, making sure you measure it accurately, understand where it is unreliable, and know how to aggregate thousands of such measurements into something that looks like the maps in the literature.

By the end of this lab you should be able to

- locate and selectively read panchromatic imagery for a named glacier without downloading entire scenes,
- implement normalized cross-correlation with a subpixel parabolic peak estimate,
- produce a velocity magnitude map and a along-flow profile, and sanity-check it against flow-law expectations from {doc}`../ice_flow/shallow-ice`,
- articulate why co-registration errors, cloud shadows, and seasonal snow dominate the error budget,
- explain, concretely, what ITS_LIVE does that one pair cannot.

```{admonition} Patterns from UW-GDA
:class: seealso
The windowed-read approach used here, opening a cloud-optimized GeoTIFF over a remote URL and fetching only the glacier sub-window rather than the full scene, follows the UW Geospatial Data Analysis course of {cite}`shean2023`, which teaches these patterns on Landsat-8 imagery through a dynamic data-access demo. Module 5 (Raster 1, rasterio, GDAL, and Landsat-8) covers the rasterio and GDAL fundamentals the image-load cells rely on, and Module 7 (warping, clipping, and DEM differencing) extends those skills to the co-registration and chip-alignment steps that make feature tracking defensible. Work through those two modules there if the tooling here feels unfamiliar.
```

## Where the images live

The two most useful panchromatic archives for glacier work are Landsat 8/9 and Sentinel-2.

**Landsat 8/9 panchromatic band (Band 8, 15 m).** NASA and USGS distribute the full archive free of charge. You can browse and order scenes at the [USGS EarthExplorer](https://earthexplorer.usgs.gov) portal, and the Level-2 and Collection-2 products are also staged as cloud-optimized GeoTIFFs on AWS open data at `s3://usgs-landsat/collection02/`. Accessing them requires no registration and no cost, though you do pay AWS egress fees if you run the download from outside `us-west-2`.

**Sentinel-2 (10 m multispectral, panchromatic-quality sharpness).** The European Space Agency's Copernicus Data Space at [dataspace.copernicus.eu](https://dataspace.copernicus.eu) hosts the full Sentinel-2 archive with a free-tier API. Programmatic access uses the OData or STAC endpoints and requires an account.

### The data-volume problem — and why you use windowed reads

A single Landsat scene covers 185 × 185 km. Its panchromatic band at 15 m resolution is a roughly 12 340 × 12 340 pixel image, which occupies about 300 MB compressed and around 1 GB when decompressed in memory. A Sentinel-2 tile at 10 m is even larger. Downloading that for two dates just to measure one glacier wastes time and disk; doing it for every scene pair in an annual mosaic is simply not feasible on a laptop.

The solution is a **windowed read via rasterio**. Cloud-optimized GeoTIFFs (COGs) store overview pyramids and tile-ordered data internally so that any rectangular sub-window can be fetched with a handful of HTTP range requests rather than the full file. `rasterio.open()` on a `/vsicurl/` or `s3://` path opens only the metadata. Then `src.read(window=...)` fetches only the tile blocks overlapping your window. For a 50 × 50 km glacier cutout you transfer a few megabytes instead of a gigabyte, and the latency is seconds rather than minutes.

### From one pair to ITS_LIVE

This lab processes exactly one pair honestly — two images, one displacement field, one velocity map. The ITS_LIVE project {cite}`mouginot2019` takes the same kernel and runs it at scale: thousands of scene pairs per glacier system, auto-QC to discard ice-free or cloud-corrupted chips, mosaicking by weighted averaging, and continuous updating as new imagery arrives. The velocity products the modeling chapters use as boundary conditions come from that pipeline. Running it yourself for one pair is the right way to understand what those products hide: the noise in any single pair, the seasonal signal averaged away by the mosaic, and the artifacts near divides where surface texture is absent. A single pair gives you intuition; ITS_LIVE gives you the number.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm
import matplotlib.ticker as ticker
from scipy.signal import fftconvolve
from scipy.ndimage import zoom

# rasterio and pyproj are used in the data-load cells.
# They are NOT imported here so that the notebook builds cleanly even
# when those cells are skipped (see the NOTE blocks below).

# ── glacier parameters for Kaskawulsh, Yukon ─────────────────────────────────
# Kaskawulsh is one of the fastest well-instrumented valley glaciers in North
# America, with surface velocities of ~1 m/d in the trunk and published
# feature-tracking records against which we can sanity-check our result.
# Surface slope ~ 1.8°, ice thickness ~ 500 m in the trunk.
GLACIER_NAME  = 'Kaskawulsh'
ICE_THICKNESS = 500.0      # m, trunk estimate
SURFACE_SLOPE = np.radians(1.8)   # radians
RHO_I         = 917.0      # kg m^-3
GRAVITY       = 9.81       # m s^-2
SPY           = 3.15576e7  # seconds per year

print('Setup complete.')print(f'Target glacier: {GLACIER_NAME}')

## Reading the image pair

We want two cloud-free Landsat 8 or 9 acquisitions of Kaskawulsh separated by roughly six months — enough time for the glacier to move tens of metres at its true velocity of ~300 m/yr, which at 15 m pixel spacing gives a sub-pixel to ~20-pixel displacement depending on location. Larger separations give more signal but risk decorrelation as crevasse geometry evolves.

The two cells below open the COGs with windowed reads and crop a 512 × 512 km sub-tile centred on the glacier trunk. The URLs point to USGS Collection-2 Level-1 panchromatic bands staged on AWS. Neither cell is executed at build time because the network request takes ~30 seconds and requires either AWS credentials or a US-west instance.

```{note}
If you run these cells, set `AWS_NO_SIGN_REQUEST=YES` in your environment (the USGS bucket allows anonymous access) or pass `aws_unsigned=True` when configuring rasterio's GDAL environment. The COG URL pattern for Collection 2 is shown in the code; verify the exact scene ID and path at EarthExplorer before running.
```

In [ ]:
# ── windowed COG read — NOT executed at build  ────────────────────────────────
#
# verify: scene IDs below are placeholders; confirm cloud-cover < 5% and
# ice-free date at EarthExplorer before substituting real IDs.
#
# import os
# import rasterio
# from rasterio.windows import from_bounds
# from rasterio.crs import CRS
#
# os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'
#
# # Landsat 9 OLI, Band 8 (panchromatic, 15 m)
# # Path/Row 061/018  covers Kaskawulsh
# SCENE_A = (
#     's3://usgs-landsat/collection02/level-1/standard/oli-tirs/2023/061/018/'
#     'LC09_L1TP_061018_20230827_20230901_02_T1/'
#     'LC09_L1TP_061018_20230827_20230901_02_T1_B8.TIF'
# )  # verify: date, scene ID, T1 tier
# SCENE_B = (
#     's3://usgs-landsat/collection02/level-1/standard/oli-tirs/2024/061/018/'
#     'LC09_L1TP_061018_20240321_20240325_02_T1/'
#     'LC09_L1TP_061018_20240321_20240325_02_T1_B8.TIF'
# )  # verify: date, scene ID, T1 tier; ~207-day separation
#
# # Bounding box of Kaskawulsh trunk in UTM zone 7N (EPSG:32607)
# BBOX = (595000, 6720000, 645000, 6770000)  # left, bottom, right, top
#
# def read_window(url, bbox, crs_epsg=32607):
#     with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
#                       CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.TIF'):
#         with rasterio.open(url) as src:
#             win = from_bounds(*bbox,
#                               transform=src.transform)
#             data = src.read(1, window=win)
#             transform = src.window_transform(win)
#     return data, transform
#
# img_a, transform_a = read_window(SCENE_A, BBOX)
# img_b, transform_b = read_window(SCENE_B, BBOX)
# DT_DAYS = 207.0   # verify from header timestamps
# PIXEL_M = abs(transform_a.a)   # pixel size in metres
# print(f'Image A shape: {img_a.shape}, pixel: {PIXEL_M:.0f} m')
# print(f'Image B shape: {img_b.shape}')
# print(f'Time separation: {DT_DAYS:.0f} days')

# ── synthetic stand-in for offline development  ──────────────────────────────
# A realistic synthetic pair: Gaussian-blurred random texture advected by a
# velocity field that has the right spatial structure for a valley glacier
# (fast trunk, slow margins).  This lets every downstream cell run without
# network access.  The inserted velocity is recoverable by the NCC tracker below.

rng = np.random.default_rng(seed=42)
NY, NX = 400, 400
PIXEL_M = 15.0
DT_DAYS = 207.0

# synthetic texture: Gaussian-blurred noise mimics crevasse/moraine contrast
from scipy.ndimage import gaussian_filter
texture = gaussian_filter(rng.standard_normal((NY, NX)).astype(np.float32), sigma=4)

# true velocity field: parabolic across-flow profile, 0 at margins
y_idx = np.arange(NY)[:, None]
x_idx = np.arange(NX)[None, :]
# glacier trunk occupies rows 100–300
trunk_mask = (y_idx >= 80) & (y_idx <= 320)
y_norm = np.where(trunk_mask, (y_idx - 200) / 120.0, 0.0)
v_true_myr = 300.0 * np.maximum(0.0, 1.0 - y_norm**2)   # m/yr, peak 300 m/yr
v_true_mday = v_true_myr / 365.25
dx_true = v_true_mday * DT_DAYS / PIXEL_M   # pixels

# advect texture by integer+fractional shift via scipy zoom trick on integer
# component; fractional remainder treated as sub-pixel linear phase
dx_int = np.round(dx_true).astype(int)
img_a = texture.copy()
img_b = np.zeros_like(texture)
for row in range(NY):
    shift = dx_int[row, 0]
    if shift < NX and shift >= 0:
        img_b[row, shift:] = texture[row, :NX - shift]
    elif shift < 0 and (-shift) < NX:
        img_b[row, :NX + shift] = texture[row, -shift:]

# add mild noise to image B to simulate radiometric differences
img_b += rng.standard_normal((NY, NX)).astype(np.float32) * 0.15

print(f'Synthetic image pair ready: {NY}×{NX} pixels, {PIXEL_M:.0f} m/pixel')
print(f'Time separation: {DT_DAYS:.0f} days')
print(f'True peak velocity: {v_true_myr.max():.0f} m/yr')

## Orienting yourself in the images

Before tracking anything it is worth spending a moment reading the images the way a field glaciologist would. The features that make cross-correlation work are exactly the features that encode the glacier's history and dynamics.

**Crevasses** open wherever the flow accelerates or turns — in the icefalls above the trunk, at the sides where marginal shear is high, and in an arc across the ablation zone where the velocity gradient transitions to slower-moving ice near the terminus. They are the best trackers: sharp, high-contrast, stable over months. The {doc}`../ice_flow/ice-flow-intro` chapter shows how ogive bands downstream of icefalls record the seasonal velocity pulse as a permanent stratigraphic marker.

**Medial moraines** are the dark stripes of debris that form where two tributary glaciers merge. They move with the flow and can be tracked over years, though their diffuse boundaries make subpixel accuracy harder than with crevasses.

**Snow-covered accumulation zones** are the worst tracking surfaces. Fresh snow buries all texture, and cross-correlation of white fields returns spurious matches everywhere. In practice, feature-tracking algorithms mask accumulation zones and report velocity only where surface texture exceeds a contrast threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
kw = dict(cmap='gray', vmin=-2.5, vmax=2.5, interpolation='none')

axes[0].imshow(img_a, **kw)
axes[0].set_title('Image A (t₁)')
axes[0].set_xlabel('x (pixels)')
axes[0].set_ylabel('y (pixels)')

axes[1].imshow(img_b, **kw)
axes[1].set_title(f'Image B (t₁ + {DT_DAYS:.0f} d)')
axes[1].set_xlabel('x (pixels)')

# Annotate approximate trunk, margin, and accumulation zones for orientation
for ax in axes:
    ax.axhline(80,  color='cyan',   lw=0.8, ls='--', alpha=0.7)
    ax.axhline(320, color='cyan',   lw=0.8, ls='--', alpha=0.7)
    ax.text(10, 50,  'accumulation / margin', color='cyan', fontsize=7, va='top')
    ax.text(10, 200, 'trunk (fast)',           color='cyan', fontsize=7, va='center')
    ax.text(10, 350, 'ablation margin',        color='cyan', fontsize=7, va='top')

plt.tight_layout()
plt.show()
print('The cyan dashed lines mark the approximate trunk boundary.')
print('Notice that the trunk texture in image B is shifted right relative to A.')

## Normalized cross-correlation from scratch

For each output point we cut a small **chip** from image A centred on that point, then search image B for the offset that makes the two patches most similar. Similarity is measured by the normalized cross-correlation (NCC) coefficient,

$$
r(\Delta x, \Delta y) = \frac{\sum_{i,j}\bigl[A(i,j) - \bar A\bigr]\bigl[B(i+\Delta y,\,j+\Delta x) - \bar B\bigr]}{\sigma_A\,\sigma_B\,N},
$$

where $\bar A$, $\bar B$ are chip means, $\sigma_A$, $\sigma_B$ are chip standard deviations, and $N$ is the number of pixels in the chip. Normalization removes the effect of brightness differences between the two images, which is important because solar angle, atmospheric haze, and sensor gain all differ between dates. $r$ runs from $-1$ to $+1$; a well-matched chip gives a sharp peak near $+1$.

Finding the integer-pixel peak is straightforward but gives velocity precision no better than one pixel. We can do much better with a **parabolic subpixel fit**: fit a parabola through the NCC surface at the integer peak and its two neighbours in each axis and take the parabola's vertex as the true peak location. For a Gaussian-shaped NCC peak the parabolic estimate recovers sub-pixel position to a fraction of a pixel, which at 15 m and 207 days corresponds to a few metres per year.

Two hyperparameters govern the trade-off explored in Task 1.

| Parameter | Small | Large |
|---|---|---|  
| **chip size** | fine spatial resolution, noisy NCC (few pixels) | smooth field, blurred gradients |
| **search window** | fast, limited to small displacements | allows large displacements, more false matches |

In [ ]:
def ncc_surface(chip_a, patch_b):
    """Compute the normalized cross-correlation surface between chip_a and patch_b.

    chip_a  : 2-D array, the template (from image A)
    patch_b : 2-D array, the search patch from image B (must be >= chip_a in each dim)

    Returns the NCC surface at each candidate integer offset.
    """
    chip_a = chip_a - chip_a.mean()
    sigma_a = chip_a.std()
    if sigma_a < 1e-6:
        return None   # featureless chip — skip
    chip_a = chip_a / sigma_a

    nc, mc = chip_a.shape   # chip dimensions
    np_, mp = patch_b.shape  # patch dimensions

    n_rows = np_ - nc + 1
    n_cols = mp - mc + 1
    surf = np.empty((n_rows, n_cols), dtype=np.float32)

    for di in range(n_rows):
        for dj in range(n_cols):
            sub = patch_b[di:di + nc, dj:dj + mc]
            sub_z = sub - sub.mean()
            sig = sub_z.std()
            if sig < 1e-6:
                surf[di, dj] = 0.0
            else:
                surf[di, dj] = np.sum(chip_a * sub_z / sig) / (nc * mc)
    return surf


def parabola_vertex(s, ri, ci):
    """Subpixel peak by parabola fit in each axis independently.

    s  : 2-D NCC surface
    ri, ci : integer row/col of the peak
    Returns (dr, dc) fractional offsets from (ri, ci).
    """
    nr, nc = s.shape
    dr = dc = 0.0
    if 0 < ri < nr - 1:
        a, b, c = s[ri - 1, ci], s[ri, ci], s[ri + 1, ci]
        denom = 2 * (a - 2 * b + c)
        if abs(denom) > 1e-10:
            dr = (a - c) / denom
    if 0 < ci < nc - 1:
        a, b, c = s[ri, ci - 1], s[ri, ci], s[ri, ci + 1]
        denom = 2 * (a - 2 * b + c)
        if abs(denom) > 1e-10:
            dc = (a - c) / denom
    return dr, dc


def track_velocity(img_a, img_b, pixel_m, dt_days,
                   chip_size=32, search_half=24, stride=16):
    """Run NCC feature tracking on an image pair and return velocity fields.

    img_a, img_b : 2-D float arrays
    pixel_m      : pixel size in metres
    dt_days      : time separation in days
    chip_size    : side length of the template chip (pixels)
    search_half  : half-width of the search window beyond chip (pixels)
    stride       : spacing between output points (pixels)

    Returns
    -------
    vx, vy   : east and north velocity components (m/yr), NaN where unreliable
    peak_r   : NCC peak value, for quality filtering
    cx, cy   : pixel coordinates of each output point
    """
    NY, NX = img_a.shape
    half_c = chip_size // 2
    half_s = half_c + search_half

    ys = np.arange(half_s, NY - half_s, stride)
    xs = np.arange(half_s, NX - half_s, stride)
    VX = np.full((len(ys), len(xs)), np.nan)
    VY = np.full((len(ys), len(xs)), np.nan)
    PKR = np.full((len(ys), len(xs)), np.nan)

    dt_yr = dt_days / 365.25
    scale = pixel_m / dt_yr   # m/yr per pixel of displacement

    for ii, cy in enumerate(ys):
        for jj, cx in enumerate(xs):
            chip = img_a[cy - half_c : cy + half_c,
                         cx - half_c : cx + half_c]
            patch = img_b[cy - half_s : cy + half_s,
                          cx - half_s : cx + half_s]
            surf = ncc_surface(chip, patch)
            if surf is None:
                continue
            ri, ci = np.unravel_index(np.argmax(surf), surf.shape)
            dr, dc = parabola_vertex(surf, ri, ci)
            # offset from centre of search window
            dy_pix = (ri + dr) - (search_half)
            dx_pix = (ci + dc) - (search_half)
            VX[ii, jj] = dx_pix * scale
            VY[ii, jj] = dy_pix * scale
            PKR[ii, jj] = surf[ri, ci]

    # mask low-correlation points (false matches)
    bad = PKR < 0.4
    VX[bad] = np.nan
    VY[bad] = np.nan

    CX, CY = np.meshgrid(xs, ys)
    return VX, VY, PKR, CX, CY


# Run with default parameters
CHIP = 32
SEARCH = 24
VX, VY, PKR, CX, CY = track_velocity(
    img_a.astype(np.float32),
    img_b.astype(np.float32),
    PIXEL_M, DT_DAYS,
    chip_size=CHIP, search_half=SEARCH, stride=16
)

VMAG = np.sqrt(VX**2 + VY**2)
n_valid = np.sum(~np.isnan(VMAG))
print(f'Tracking complete: {n_valid} valid vectors out of {VMAG.size}')
print(f'Median speed: {np.nanmedian(VMAG):.0f} m/yr')
print(f'Max speed:    {np.nanmax(VMAG):.0f} m/yr')

## Velocity field and profile

Two representations are useful. The **magnitude map** shows the spatial pattern at a glance — fast trunk, slow margins, the speed-up through icefalls. A **along-flow profile** extracts quantitative numbers and is the right thing to compare against model output or against the ITS_LIVE mosaics.

The SIA (shallow-ice approximation, {doc}`../ice_flow/shallow-ice`) gives us an independent bound to check against. For deformation alone, the surface velocity predicted by Glen's flow law integrated over depth is

$$
u_s = \frac{2A}{n+1}\,(\rho_i\,g\,\sin\alpha)^n\,H^{n+1},
$$

with $A \approx 2.4\times 10^{-24}\,\mathrm{Pa}^{-3}\,\mathrm{s}^{-1}$ at $-5^\circ\mathrm{C}$ and $n = 3$. Kaskawulsh slides, so the measured surface velocity should exceed the SIA deformation prediction — the difference is the basal sliding contribution. A measured velocity well below the SIA prediction would signal a tracking error.

In [ ]:
# ── SIA deformation velocity as a sanity-check reference ─────────────────────
A_glen = 2.4e-24     # Pa^-3 s^-1 at approx -5 C  [TODO-CITE: cuffey2010]
n_glen = 3.0
tau_d  = RHO_I * GRAVITY * ICE_THICKNESS * np.sin(SURFACE_SLOPE)   # Pa
u_def_si = (2 * A_glen / (n_glen + 1)) * tau_d**n_glen * ICE_THICKNESS   # m/s
u_def_myr = u_def_si * SPY   # m/yr
print(f'SIA deformation speed (trunk centre): {u_def_myr:.0f} m/yr')
print(f'Measured peak speed (NCC):            {np.nanmax(VMAG):.0f} m/yr')
excess = np.nanmax(VMAG) - u_def_myr
print(f'Excess attributed to sliding:         {excess:.0f} m/yr')
if excess < 0:
    print('  Note: negative excess — either the chip parameters need tuning,'
          ' or the SIA parameters are off for this location.')

# ── magnitude map ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.imshow(img_a, cmap='gray', vmin=-2.5, vmax=2.5,
          extent=[0, NX * PIXEL_M / 1e3, NY * PIXEL_M / 1e3, 0],
          interpolation='none', alpha=0.6)
speed_plot = ax.pcolormesh(
    CX * PIXEL_M / 1e3, CY * PIXEL_M / 1e3, VMAG,
    cmap='plasma', vmin=0, vmax=350, alpha=0.85
)
cb = plt.colorbar(speed_plot, ax=ax, fraction=0.03, pad=0.04)
cb.set_label('Speed (m/yr)')
ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')
ax.set_title(f'{GLACIER_NAME}: velocity magnitude')
ax.axhline(80 * PIXEL_M / 1e3,  color='cyan', lw=0.7, ls='--')
ax.axhline(320 * PIXEL_M / 1e3, color='cyan', lw=0.7, ls='--')

# ── across-flow profile at mid-glacier ───────────────────────────────────────
ax2 = axes[1]
mid_row = np.argmin(np.abs(CY[:, 0] - 200))   # closest row to y=200 px
profile_x = CX[mid_row, :] * PIXEL_M / 1e3
profile_v = VMAG[mid_row, :]
ax2.plot(profile_x, profile_v, 'b-o', ms=3, lw=1.5, label='NCC measured')
ax2.axhline(u_def_myr, color='k', ls='--', lw=1,
            label=f'SIA deformation ({u_def_myr:.0f} m/yr)')
ax2.set_xlabel('x (km)')
ax2.set_ylabel('Speed (m/yr)')
ax2.set_title('Across-flow profile (mid-glacier)')
ax2.legend(fontsize=9)
ax2.set_ylim(0, None)

plt.tight_layout()
plt.show()

## Error sources

A velocity map from NCC is only as good as its error budget. Three sources dominate in practice, and understanding them is what separates a publishable result from a pretty-looking artefact.

### Co-registration error

The two images must be in exactly the same geographic coordinate system before any chip is correlated. Even a half-pixel systematic offset between the two scenes — caused by orbital uncertainty, atmospheric refraction, terrain relief displacement, or geolocation error in the level-1 product — propagates into every velocity vector as a spatially coherent bias. At 15 m pixel size and a 207-day separation, half a pixel is 26 m/yr, which is comparable to the velocity of a slow alpine glacier. Standard practice is to co-register the pair by correlating stable, non-moving areas (bedrock outcrops, moraines beyond the glacier margin) and subtracting the mean residual offset from the velocity field.

### False correlation on clouds and shadows

A cloud in image A and a cloud in image B may happen to match each other even though they are unrelated. The NCC coefficient will be high — cloud texture is real texture — but the inferred displacement is nonsense. Cloud masks from the scene metadata help, but optically thin cirrus and patchy shadow fields are the hardest cases. Filtering by NCC peak value (as done above with the `peak_r < 0.4` threshold) catches some false matches, but not all. Comparing the velocity against a neighbouring season's pair is the most reliable quality check.

### Decorrelation over fresh snow

When fresh snow buries the accumulation zone between the two acquisitions, every chip from image A finds zero contrast in image B and the NCC surface is flat rather than peaked. The parabolic vertex finder returns a noise-driven artefact, not a real displacement. The `sigma < 1e-6` guard in `ncc_surface` rejects perfectly uniform chips, but real snow fields have just enough sensor noise to pass that test. The practical remedy is to mask the accumulation zone with a snow-cover product or to use only late-summer imagery when glacier ice is exposed as far up-glacier as possible. This is one reason ITS_LIVE favours late-summer pairs for its primary velocity layers.

## Task 1: chip size and search-window trade-offs

The two hyperparameters of the tracker interact in a way that is not obvious until you try both extremes.

Run the tracker three times, varying only `chip_size` while keeping `search_half = 24` and `stride = 16`. Try chip sizes of 16, 32, and 64 pixels. For each run, plot the magnitude map and report the median speed, the number of valid vectors, and the standard deviation of the speed in the known-fast trunk region (rows 80–320, all columns). Then answer the following.

1. For the 16-pixel chip, is the speed in the trunk higher or lower than for the 32-pixel chip, and is the scatter higher or lower? Explain both in terms of the NCC surface shape.
2. For the 64-pixel chip, what happens to the sharpness of the velocity gradient at the trunk margin? Why does a large chip smooth that gradient?
3. At what chip size would you trust a velocity estimate within 5 m/yr, assuming the NCC peak is Gaussian-shaped and the image noise level is ~10% of the signal standard deviation? You do not need to derive the formula from scratch; reason from the shape of the parabolic fit error.

In [ ]:
chip_sizes = [16, 32, 64]
results = {}

for cs in chip_sizes:
    vx, vy, pkr, cx, cy = track_velocity(
        img_a.astype(np.float32),
        img_b.astype(np.float32),
        PIXEL_M, DT_DAYS,
        chip_size=cs, search_half=24, stride=16
    )
    vmag = np.sqrt(vx**2 + vy**2)
    # trunk mask in output-pixel coordinates
    trunk_rows = (cy >= 80) & (cy <= 320)
    trunk_v = vmag[trunk_rows]
    results[cs] = dict(vmag=vmag, cx=cx, cy=cy,
                       median=np.nanmedian(vmag),
                       n_valid=np.sum(~np.isnan(vmag)),
                       trunk_std=np.nanstd(trunk_v))
    print(f'chip={cs:2d}: median={results[cs]["median"]:6.0f} m/yr  '
          f'valid={results[cs]["n_valid"]:5d}  '
          f'trunk_std={results[cs]["trunk_std"]:5.0f} m/yr')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, cs in zip(axes, chip_sizes):
    r = results[cs]
    im = ax.pcolormesh(r['cx'] * PIXEL_M / 1e3, r['cy'] * PIXEL_M / 1e3, r['vmag'],
                       cmap='plasma', vmin=0, vmax=350)
    ax.set_title(f'chip = {cs} px')
    ax.set_xlabel('x (km)')
axes[0].set_ylabel('y (km)')
plt.colorbar(im, ax=axes[-1], label='Speed (m/yr)', fraction=0.03)
plt.suptitle('Task 1: chip-size sensitivity', y=1.01)
plt.tight_layout()
plt.show()

# YOUR WRITTEN ANSWERS HERE (add a markdown cell below or write as comments):
# 1. chip=16 vs chip=32: speed and scatter comparison
# 2. chip=64 margin sharpness
# 3. chip size for 5 m/yr precision

## Task 2: time-separation trade-off

The synthetic pair was built with `DT_DAYS = 207`. Repeat the tracking for time separations of 30, 90, 207, and 365 days by rescaling the true displacement field used to build synthetic image B, then re-running the tracker with `chip_size=32, search_half=24`. For each separation, compute the median absolute error between the recovered and true velocity magnitude in the trunk region.

Answer the following.

1. At 30-day separation, the displacement of a 300 m/yr point is about 25 m, or roughly 1.6 pixels. What goes wrong with the parabolic subpixel estimate when the true displacement is close to one pixel? Sketch the expected shape of the NCC surface for this case.
2. At 365-day separation, the crevasse field has evolved as the glacier flowed. In the synthetic case that is not modelled — real images would decorrelate. State two physical processes that cause a crevasse to change its appearance over a year, and say which one would be worse for a glacier accelerating in winter.
3. For a glacier moving at 50 m/yr (typical of a slow Greenland outlet that is not surging), what time separation would you choose to get a 2-pixel displacement at 15 m resolution, and is that a scientifically useful minimum?

In [ ]:
dt_values = [30, 90, 207, 365]
errors = []

for dt in dt_values:
    # rebuild image B for this separation
    v_mday_loc = v_true_myr / 365.25
    dx_t = v_mday_loc * dt / PIXEL_M
    dx_int_t = np.round(dx_t).astype(int)
    img_b_t = np.zeros_like(texture)
    for row in range(NY):
        shift = dx_int_t[row, 0]
        if 0 <= shift < NX:
            img_b_t[row, shift:] = texture[row, :NX - shift]
        elif shift < 0 and -shift < NX:
            img_b_t[row, :NX + shift] = texture[row, -shift:]
    img_b_t += rng.standard_normal((NY, NX)).astype(np.float32) * 0.15

    vx_t, vy_t, _, cx_t, cy_t = track_velocity(
        img_a.astype(np.float32), img_b_t.astype(np.float32),
        PIXEL_M, dt, chip_size=32, search_half=24, stride=16
    )
    vmag_t = np.sqrt(vx_t**2 + vy_t**2)

    # true velocity at output points
    v_true_at_pts = v_true_myr[cy_t, cx_t]
    err = np.nanmedian(np.abs(vmag_t - v_true_at_pts))
    errors.append(err)
    print(f'dt={dt:4d} d  median |error| = {err:.0f} m/yr  '
          f'n_valid={np.sum(~np.isnan(vmag_t))}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(dt_values, errors, 'o-', lw=2)
ax.set_xlabel('Time separation (days)')
ax.set_ylabel('Median |velocity error| (m/yr)')
ax.set_title('Task 2: time-separation sensitivity')
ax.xaxis.set_major_locator(ticker.MultipleLocator(60))
plt.tight_layout()
plt.show()

# YOUR WRITTEN ANSWERS HERE:
# 1. NCC surface shape at ~1-pixel displacement
# 2. Two processes causing crevasse appearance change
# 3. Minimum useful separation for 50 m/yr glacier

## Synthesis: from one pair to ITS_LIVE-style mosaics

You have now run the NCC kernel, seen its two main hyperparameters, and measured its error budget in a controlled synthetic setting. Let us step back and ask what the gap is between what you did and the velocity products that the ice-sheet modelling chapters use as boundary conditions.

**One pair gives one snapshot in time, with unknown phase.** The velocity on any glacier varies seasonally — Kaskawulsh is faster in summer when melt-water pressurises the bed — and episodically during surges {cite}`kamb1987`. A single pair captures one sample from that distribution. Two pairs from different seasons can disagree by a factor of two at the same location and both be correct. ITS_LIVE takes every available Landsat 8/9 pair over a region, runs the tracker on each, and builds a temporal mean with uncertainty quantified from the spread across pairs. That temporal averaging is most of what turns a noisy snapshot into a number a model can use.

**The co-registration bias is the largest systematic error.** As argued above, half-pixel geolocation error produces a 20–30 m/yr bias at Landsat resolution — larger than the signal from a slow outlet glacier. ITS_LIVE and similar products (GoLIVE, ITSLIVE-V2) correct this by correlating stable off-ice terrain in every pair; the lab's synthetic images skip that step because the terrain does not move. On real data you would add a `coregister_pair()` function before calling `track_velocity()`, using bedrock outcrops as ground-control chips.

**What the mosaics hide.** The modelling chapters' velocity boundary conditions are annual means, usually from the 2017–2019 ITS_LIVE v2 mosaic {cite}`mouginot2019`. Three things are folded in that you cannot see.

1. The seasonal signal: the annual mean velocity is not the same as the summer velocity that the diagnostic-inversion chapters use to infer basal friction. A fast-sliding summer and a slow winter average to something that belongs to neither.
2. The elevation-change signal: as a glacier thins, its driving stress drops, and the velocity from 2017 is not the velocity for a thinned-down 2050 configuration. Flow models that use a fixed velocity field are making a steady-state assumption that breaks down for rapidly changing glaciers.
3. The spatial smoothing from the mosaic's gridding: the 240 m ITS_LIVE grid smooths across the sharp velocity gradients at shear margins, which are exactly where the stress-balance inversions are most sensitive. Subpixel accuracy at the chip level does not survive coarse gridding of the mosaic.

None of this makes ITS_LIVE wrong or unusable — it is the best large-scale observational product available and the chapter's flow-law chapters could not have been written without it. But running one pair yourself, and measuring its errors, is what lets you read the uncertainty bands on those products honestly.

## Synthesis questions

Write a short paragraph — or answer point by point — addressing the following.

1. The SIA velocity computed above uses only surface slope and ice thickness. You found that the measured velocity exceeds the SIA prediction. State one thing that the SIA ignores that explains the excess, and say what observation you would need to quantify it independently.

2. In Task 2 you found that short time separations give noisy velocity estimates. But the ITS_LIVE algorithm actually uses short separations preferentially for fast-moving outlet glaciers. Why might a 16-day Sentinel-2 pair be more useful than a 365-day Landsat pair for a glacier accelerating across a drainage-basin divide?

3. The error-source section listed co-registration, cloud/shadow false matches, and snow decorrelation. Rank these three by the difficulty of correcting them automatically in a production pipeline (easiest to hardest), and justify your ranking in one sentence each.

4. The diagnostic-inversion chapters ({doc}`../thermomechanics/basal-motion`) use surface velocity as the target observable to infer basal friction. If the velocity product you feed into the inversion carries a co-registration bias of 25 m/yr, how large an error in basal friction does that imply for a glacier with a sliding-law sensitivity of roughly $\partial u / \partial \beta \approx 10\,\mathrm{m\,yr^{-1}\,per\,kPa}$? Is that acceptable?